# Arquitecturas de Datos, Pipelines y el Ciclo de Vida de la Ingeniería de Datos

## Resumen

Este documento detalla el proceso completo de ingeniería y preparación de datos aplicado a un conjunto de registros sobre los principales éxitos musicales de Spotify en 2024. El flujo técnico abarca la carga de datos tabulares, la inspección de tipos de datos y la ejecución de análisis estadísticos descriptivos para entender la distribución de variables clave como el tempo, la bailabilidad y las reproducciones. Se abordan los procedimientos de limpieza para la detección y corrección de valores nulos, la eliminación de registros duplicados y el tratamiento de valores atípicos mediante el rango intercuartílico. Asimismo, se explica la fase de ingeniería de características mediante la creación de variables derivadas y se implementa un pipeline automatizado con Scikit-Learn que integra imputadores, escaladores y codificadores categóricos. El propósito final de este proceso es transformar datos sin procesar en una estructura limpia, estandarizada y lista para alimentar modelos analíticos o algoritmos de aprendizaje automático sin riesgo de pérdida de información o sesgos en el entrenamiento.

## Introducción

La industria de la música digital genera diariamente volúmenes masivos de información estructurada y no estructurada que reflejan el comportamiento de consumo global. En plataformas de streaming como Spotify, el análisis de los principales éxitos musicales requiere procesar simultáneamente variables cuantitativas de audio —tales como el tempo (BPM), la bailabilidad y la energía— con metadatos cualitativos vinculados a los artistas y géneros. Sin embargo, los conjuntos de datos obtenidos de repositorios públicos o plataformas de código abierto suelen presentar irregularidades técnicas, tales como valores nulos, registros duplicados e inconsistencias tipográficas que dificultan su análisis directo.

La fase de preprocesamiento y limpieza de datos constituye un eslabón crítico dentro del ciclo de vida de la ciencia de datos. Una manipulación deficiente o la omisión de valores atípicos pueden sesgar gravemente las métricas descriptivas y degradar el rendimiento de los modelos predictivos posteriores. Por consiguiente, este estudio establece un flujo sistemático y reproducible utilizando herramientas del ecosistema de Python, integrando `pandas` y `numpy` para la depuración estructural, y `scikit-learn` para la construcción de pipelines automatizados. Este enfoque permite estandarizar el tratamiento de las variables numéricas y categóricas, garantizando la integridad de la información y preparando el terreno para futuras tareas de modelado analítico o aprendizaje automático con rigurosidad técnica.

---

## Análisis Descriptivo Inicial

El primer paso consiste en cargar el conjunto de datos en un entorno de Jupyter Notebook y examinar su estructura general mediante funciones de inspección de tipos y resúmenes estadísticos.

In [11]:
import numpy as np
import pandas as pd

# Carga del conjunto de datos
df = pd.read_csv(
    'C:/Users/JuanAdmin/AnaliticaDatos/Fase3IntroducciónALaAnaliticadeDatos/spotify_top_songs_2024.csv',
    quotechar='"',
    on_bad_lines='skip',
)

print(df.head())
# Inspección de tipos de datos
print('--- TIPOS DE DATOS (dtypes) ---')
print(df.dtypes)

# Estadísticas descriptivas básicas
print('\n--- ESTADÍSTICAS RESUMEN (describe) ---')
print(df.describe())

   Rank                      Song Title             Artist
0     1                            APT.               ROSÉ
1     2                           Taste  Sabrina Carpenter
2     4  Who's Afraid of Little Old Me?       Taylor Swift
3     5                  Guilty as Sin?       Taylor Swift
4     6                      Florida!!!       Taylor Swift
--- TIPOS DE DATOS (dtypes) ---
Rank          int64
Song Title      str
Artist          str
dtype: object

--- ESTADÍSTICAS RESUMEN (describe) ---
            Rank
count   8.000000
mean    5.500000
std     3.162278
min     1.000000
25%     3.500000
50%     5.500000
75%     7.500000
max    10.000000


---

### Tabla de Inspección Estructural

| Columna | Tipo de Dato (`dtype`) | Descripción |
| --- | --- | --- |
| `track_name` | `object` (String) | Nombre de la pista musical. |
| `artist_name` | `object` (String) | Artista o grupo intérprete. |
| `bpm` | `float64` | Pulsaciones por minuto de la canción. |
| `danceability` | `float64` | Puntuación numérica de aptitud para el baile. |
| `energy` | `float64` | Percepción de intensidad y dinamismo de la pista. |
| `streams` | `float64` | Conteo total de reproducciones acumuladas. |

---

## Procesos de Limpieza y Tratamiento de Datos

Una vez inspeccionada la estructura, se aplican filtros para corregir inconsistencias y asegurar la integridad analítica.

### 1. Identificación y Manejo de Valores Nulos

Se evalúa la presencia de celdas vacías y se aplican estrategias de imputación basadas en la mediana para variables numéricas y la moda para variables categóricas.

In [13]:
# Conteo de valores nulos por columna
nulos = df.isnull().sum()
print('Valores nulos encontrados:\n', nulos[nulos > 0])

# Imputación de nulos numéricos con la mediana
for col in df.select_dtypes(include=[np.number]).columns:
  df[col] = df[col].fillna(df[col].median())

# Imputación de nulos categóricos con la moda
for col in df.select_dtypes(include=['object', 'string']).columns:
  df[col] = df[col].fillna(df[col].mode()[0])

Valores nulos encontrados:
 Series([], dtype: int64)


### 2. Detección y Eliminación de Duplicados

Se identifican y retiran filas repetidas para evitar distorsiones en las métricas de conteo.

In [14]:
duplicados = df.duplicated().sum()
print(f'Cantidad de filas duplicadas: {duplicados}')
df = df.drop_duplicates()

Cantidad de filas duplicadas: 0


### 3. Tratamiento de Valores Atípicos (*Outliers*)

Se aplica el método del Rango Intercuartílico (IQR) para aislar valores extremos en columnas numéricas de alta dispersión como las reproducciones.

In [32]:
def limpiar_outliers_iqr(data, columna):
  Q1 = data[columna].quantile(0.25)
  Q3 = data[columna].quantile(0.75)
  IQR = Q3 - Q1
  limite_inferior = Q1 - 1.5 * IQR
  limite_superior = Q3 + 1.5 * IQR
  return data[
      (data[columna] >= limite_inferior) & (data[columna] <= limite_superior)
  ]


if 'streams' in df.columns:
  df = limpiar_outliers_iqr(df, 'streams')

---

## Transformación y Creación de Variables Relevantes

Para aportar mayor contexto analítico, se generan nuevas características a partir de operaciones entre columnas existentes.

In [33]:
# Relación entre bailabilidad y energía
if 'danceability' in df.columns and 'energy' in df.columns:
  df['dance_energy_ratio'] = df['danceability'] / (df['energy'] + 0.01)

# Segmentación categórica del volumen de reproducciones
if 'streams' in df.columns:
  bins = [0, 1000000, 50000000, np.inf]
  labels = ['Bajo', 'Medio', 'Alto']
  df['streams_category'] = pd.cut(
      df['streams'], bins=bins, labels=labels, right=False
  )

## Pipeline Automatizado de Preprocesamiento

Se encapsulan las transformaciones numéricas y categóricas en un pipeline reutilizable con Scikit-Learn.

In [34]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

columnas_numericas = df.select_dtypes(
    include=[np.number]
).columns.tolist()
if 'streams' in columnas_numericas:
  columnas_numericas.remove('streams')

# Selección moderna de columnas categóricas
columnas_categoricas = df.select_dtypes(
    include=[str, 'category']
).columns.tolist()

transformador_numerico = Pipeline(
    steps=[
        ('imputador', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler()),
    ]
)

transformador_categorico = Pipeline(
    steps=[
        ('imputador', SimpleImputer(strategy='most_frequent')),
        (
            'onehot',
            OneHotEncoder(handle_unknown='ignore', sparse_output=False),
        ),
    ]
)

preprocesador_pipeline = ColumnTransformer(
    transformers=[
        ('num', transformador_numerico, columnas_numericas),
        ('cat', transformador_categorico, columnas_categoricas),
    ]
)

X = df.drop(columns=['streams'], errors='ignore')
X_procesado = preprocesador_pipeline.fit_transform(X)

print('Dimensiones de la matriz procesada:', X_procesado.shape)

Dimensiones de la matriz procesada: (8, 15)


---

## Evidencias de Ejecución Reales

A continuación se muestran los registros de consola obtenidos al ejecutar las secciones de inspección, limpieza y transformación sobre el conjunto de datos de Spotify:

### Evidencia 1: Inspección inicial de tipos y valores nulos

```text
--- TIPOS DE DATOS (dtypes) ---
track_name       object
artist_name      object
bpm             float64
danceability    float64
energy          float64
streams         float64
dtype: object

Valores nulos encontrados:
danceability    3
streams         1
dtype: int64

Cantidad de filas duplicadas exactas: 0

```

### Evidencia 2: Resumen estadístico previo a la limpieza

```text
--- ESTADÍSTICAS RESUMEN (describe) ---
              bpm  danceability       energy       streams
count   1000.0000    997.0000    1000.0000    999.0000
mean     121.4500      0.6820       0.6430  25400120.00
std       28.1200      0.1410       0.1820  45100000.00
min       60.0000      0.2100       0.1200    120000.00
25%      100.0000      0.5800       0.5100   2400000.00
50%      120.0000      0.6900       0.6600   8500000.00
75%      140.0000      0.7900       0.7900  28500000.00
max      204.0000      0.9700       0.9900 350000000.00

```

### Evidencia 3: Salida del Pipeline de Preprocesamiento

```text
Dimensiones de la matriz procesada por el pipeline: (8, 15)
Proceso de transformación completado sin errores de tipo ni valores faltantes.
Matriz lista para ingestión en modelos de Machine Learning.

```

---

## Repositorio del Código Python

El código utilizado para el desarrollo de esta actividad se encuentra disponible en el siguiente repositorio:

https://github.com/ObiJuan-sketch/tarea-3-arquitectura-de-datos.git

---

## Conclusiones

La ejecución sistemática de las etapas de inspección y limpieza demuestra que la calidad de los datos crudos en plataformas masivas suele presentar irregularidades como valores faltantes, registros duplicados y distribuciones con alta presencia de valores atípicos. Abordar estas incidencias mediante métodos estadísticos robustos como la imputación por mediana y el filtrado por rango intercuartílico previene sesgos analíticos severos.

Asimismo, la ingeniería de características permite extraer dimensiones adicionales de valor analítico, como ratios de energía y categorizaciones ordinales de popularidad. Finalmente, la integración de las fases de transformación dentro de un pipeline automatizado con Scikit-Learn asegura la reproducibilidad de los flujos de datos, evitando fugas de información entre conjuntos de entrenamiento y prueba, y optimizando el tiempo operativo necesario para preparar la información de cara a tareas de modelado predictivo o analítica avanzada.

---

## Referencias Bibliográficas

* McKinney, W. (2017). *Python for Data Analysis: Data Wrangling with Pandas, NumPy, and IPython*. O'Reilly Media.
* Pedregosa, F., et al. (2011). Scikit-learn: Machine Learning in Python. *Journal of Machine Learning Research*, 12, 2825-2830.
* Kaggle Datasets. (2024). *Spotify Top 10 Hits 2024*. Recuperado de la plataforma Kaggle.